# Week 9, Day 4: The Hessian: curvature, minima and saddle points

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w009/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Numerical Hessians and critical points

For $f(x, y) = x^3 - 3x + y^2$, write `numerical_hessian(f, w, h=1e-4)` from central differences of the gradient (use `grad_f`). Both (1, 0) and (−1, 0) are critical points; classify each from the eigenvalues of the Hessian, storing 'minimum', 'maximum' or 'saddle' in the dictionary `kinds`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
def f(w): return w[0] ** 3 - 3 * w[0] + w[1] ** 2
def grad_f(w): return np.array([3 * w[0] ** 2 - 3, 2 * w[1]])

def numerical_hessian(f, w, h=1e-4):
    n = len(w)
    H = np.zeros((n, n))
    for j in range(n):
        e = np.zeros(n); e[j] = h
        H[:, j] = ...
    return 0.5 * (H + H.T)               # symmetrise to remove rounding asymmetry

def kind(H):
    ev = np.linalg.eigvalsh(H)
    ...

kinds = {name: kind(numerical_hessian(f, np.array(p, dtype=float))) for name, p in [("(1, 0)", (1, 0)), ("(-1, 0)", (-1, 0))]}
print(kinds)

In [ ]:
import numpy as np
def test_hess():
    "numerical_hessian matches the analytic Hessian of f"
    for w in [np.array([1.0, 0.0]), np.array([0.3, -0.7])]:
        Ha = np.array([[6 * w[0], 0.0], [0.0, 2.0]])
        assert np.allclose(numerical_hessian(f, w), Ha, atol=1e-4)

def test_kinds():
    "kinds labels (1, 0) a minimum and (-1, 0) a saddle"
    assert kinds["(1, 0)"] == "minimum" and kinds["(-1, 0)"] == "saddle"

test_hess()
test_kinds()
print("All checks passed")

### Exercise: How condition number slows descent

Write `steps_needed(H, lr)`: start at (1, 1), run gradient descent on $\frac{1}{2}\mathbf{w}^\top H\mathbf{w}$ and count the steps until the length of $\mathbf{w}$ falls below 1e-3 (stop at 100,000). For Hessians with eigenvalues (1, 1), (10, 1) and (100, 1), use a learning rate of $1/\lambda_{\max}$ and store the counts in `steps`.

In [ ]:
def steps_needed(H, lr, tol=1e-3, max_steps=100000):
    w = np.array([1.0, 1.0])
    for k in range(max_steps):
        if np.linalg.norm(w) < tol:
            return k
        w = ...
    return max_steps

steps = []
for big in [1.0, 10.0, 100.0]:
    H = np.diag([big, 1.0])
    steps.append(steps_needed(H, ...))
print("condition numbers 1, 10, 100 -> steps", steps)

In [ ]:
import numpy as np
def test_steps():
    "steps_needed counts steps until the distance to the minimum falls below 1e-3"
    assert steps_needed(np.diag([1.0, 1.0]), 0.9) < 10

def test_scaling():
    "with the best stable learning rate, steps grow roughly in proportion to the condition number"
    assert steps[0] <= 2 and steps[1] > 30
    assert 5 < steps[2] / steps[1] < 20

test_steps()
test_scaling()
print("All checks passed")

### Exercise: Newton's method for logistic regression

For the logistic model of last Saturday ($p = \sigma(Z\mathbf{w})$, cross-entropy loss), the Hessian is $H = \frac{1}{T}Z^\top\operatorname{diag}(\mathbf{p}(1 - \mathbf{p}))Z$. Write `hess_xent(w)`, then run 10 Newton steps and 100 gradient-descent steps (learning rate 1) from zero, recording the loss after each step in `newton_losses` and `gd_losses`.

In [ ]:
rng = np.random.default_rng(14)
T = 400
Z = np.column_stack([rng.normal(size=(T, 2)) @ np.array([[1.0, 0.9], [0.0, 0.45]]), np.ones(T)])   # correlated inputs plus bias
w_true = np.array([2.0, -1.5, 0.3])
y = (rng.random(T) < 1 / (1 + np.exp(-Z @ w_true))).astype(float)
def sigmoid(u): return 1 / (1 + np.exp(-u))
def xent(w):
    p = sigmoid(Z @ w); return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
def grad_xent(w): return Z.T @ (sigmoid(Z @ w) - y) / T

def hess_xent(w):
    p = sigmoid(Z @ w)
    return ...

w = np.zeros(3); newton_losses = []
for _ in range(10):
    w = ...
    newton_losses.append(xent(w))
v = np.zeros(3); gd_losses = []
for _ in range(100):
    v = v - 1.0 * grad_xent(v)
    gd_losses.append(xent(v))
best = newton_losses[-1]

plt.figure(figsize=(5, 3)); plt.semilogy(np.array(newton_losses) - best + 1e-16, "o-", label="Newton"); plt.semilogy(np.array(gd_losses) - best + 1e-16, label="gradient descent")
plt.xlabel("step"); plt.ylabel("loss above the best"); plt.legend(frameon=False); plt.show()
print("Newton:", np.round(newton_losses[:5], 6), " GD after 100 steps:", round(gd_losses[-1], 6))

In [ ]:
import numpy as np
def test_hessian():
    "hess_xent matches the numerical Hessian of the cross-entropy"
    w = np.random.default_rng(0).normal(size=3) * 0.3
    g = lambda v: grad_xent(v)
    Hn = np.zeros((3, 3))
    for j in range(3):
        e = np.zeros(3); e[j] = 1e-5
        Hn[:, j] = (g(w + e) - g(w - e)) / 2e-5
    assert np.allclose(hess_xent(w), Hn, atol=1e-6)

def test_fast():
    "Newton gets within 1e-8 of the best loss in at most 10 steps, while gradient descent is still far off after 100"
    assert newton_losses[9] - best < 1e-8
    assert gd_losses[99] - best > 1e-4

test_hessian()
test_fast()
print("All checks passed")